In [44]:
#!pip install pandapower pandas nbformat

## Tratamento de dados de arquivos de texto do CEPEL para ANAREDE

In [45]:
import pandas as pd
import re

def read_dataset_anarede(filepath):
    with open(filepath, encoding="utf-8") as f:
        lines = f.readlines()

    section_pattern = re.compile(r"^(DBAR|DLIN|DGLT|DARE|DGBT)\s*$")
    end_pattern = re.compile(r"^99999")
    tables = {}
    current_section = None
    current_header = []
    current_data = []

    def save_table():
        if current_section and current_header and current_data:
            adjusted_data = []
            for row in current_data:
                if len(row) < len(current_header):
                    row = row + [""] * (len(current_header) - len(row))
                elif len(row) > len(current_header):
                    row = row[:len(current_header)]
                adjusted_data.append(row)
            df = pd.DataFrame(adjusted_data, columns=current_header)
            tables[current_section] = df

    for line in lines:
        line = line.rstrip("\n")
        section_match = section_pattern.match(line)
        if section_match:
            save_table()
            current_section = section_match.group(1)
            current_header = []
            current_data = []
            continue

        if current_section:
            if not current_header and line.startswith("("):
                header = re.findall(r"\(([^)]+)\)", line)
                current_header = [h.strip() for h in header]
                continue
            if end_pattern.match(line):
                save_table()
                current_section = None
                current_header = []
                current_data = []
                continue
            if line.strip() and not line.startswith("("):
                row = re.split(r'\s{2,}', line.strip())
                current_data.append(row)

    # Salva a última tabela se necessário
    save_table()
    return tables



# Exemplo de uso:
print("Lendo o arquivo ANAREDE de sistema de 16 Barras com Fluxo FBA 100...")
dfs = read_dataset_anarede(r"C:\Users\pedrovictor.veras\Documents\GitHub\Repopulation-With-Elite-Set\src\RedeEletrica\assets\arquivos_anarede\CASO_FBA100.txt")

print("Tabelas lidas:")
for key, df in dfs.items():
    print(f"{key}:")
    display(df.head(), "\n")  # Exibe as primeiras linhas de cada DataFrame
    

print("\nTabela de Barras")
#display(dfs["DBAR"])
print(dfs["DBAR"]["nome"].unique())
#display(dfs["DLIN"])

Lendo o arquivo ANAREDE de sistema de 16 Barras com Fluxo FBA 100...
Tabelas lidas:
DBAR:


,Num,nome,V,A,Pg,Qg,Qn,Qm,Bc,Pl,Ql,Sh,Vf
0,10 L2 FCANARIO-18,1015,0.383.3-12.4-140. 140.,11030,,,,,,,,,
1,11 L1 FSABIA---13,1015-12. 250.12.43-100. 100.,11030,,,,,,,,,,
2,20 L1 FTUCANO--13,9804.18 373.-33.3-140. 140.,21030,,,,,,,,,,
3,21 L1 FGAVIAO--13,9601.52 250.-36.9 -90.,90.,21030,,,,,,,,,
4,100 L,BCANARIO230,11023-5.6,11000,,,,,,,,,


'\n'

DLIN:


,De,Pa,R%,X%,Mvar,Tap,Tmn,Tmx,Phs,Bc,Cn,Ce
0,10,100 1,2.658,1.,,,,,,,,
1,11,110 1,3.8525,1.,,,,,,,,
2,20,200 1,2.725,1.,,,,,,,,
3,21,210 1,3.44,1.,,,,,,,,
4,100,120 1,2.76 10.44 18.43,,,,,,,,,


...

'\n'

DGBT:


,G ( kV
0,A
1,B
2,C
3,D
4,E


'\n'


Tabela de Barras
['1015' '1015-12. 250.12.43-100. 100.' '9804.18 373.-33.3-140. 140.'
 '9601.52 250.-36.9 -90.' 'BCANARIO230' 'BSABIA--230' 'CSABIA--138'
 'ASABIA--440' 'FSABIA---13' 'BCARDEAL230' 'DCARDEAL-88' 'BCURIO--230'
 'ACURIO--CS5' 'ACURIO--440' 'FCURIO---13' 'ECURIO---69' 'CCURIO--138'
 'ESANHACO-69' 'BTIZIU--230' 'CTIZIU--138' 'CPARDAL-138' 'EPARDAL--69'
 'CAZULAO-138' 'EAZULAO--69' 'ABICUDO-440' 'EBICUDO--69' 'FBICUDO--13'
 'ACHOPIN-440' 'CCHOPIN-138' 'BTUCANO-230' 'BGAVIAO-230' 'BARARA--230'
 'AARARA--CS5' 'AARARA--440' 'FARARA---13' 'CARARA--138' 'BPELICAN230'
 'APELICAN440' 'FPELICANO13' 'BCORUJA-230' 'BURUBU--230' 'CURUBU--138'
 'BGARCA--230' 'FSABIA--FIC' 'FCURIO--FIC' 'FBICUDO-FIC' 'FARARA--FIC'
 'FPELICANFIC']


## Analise de Contigencias com dados do AnaREDE


### Estrutura do Arquivo CSV:

O arquivo contém colunas bem organizadas que representam um mix de dados de DBAR, DLIN, e até campos típicos de fluxos de potência.

Colunas principais encontradas:

- Barra → Número da barra

- Nome → Nome da barra (exemplo: Canário, Tucano, etc)

- Vbase_kV → Tensão base da barra (em kV)

- Pger_MW, Qger_Mvar → Potência gerada (se houver)

- Pload_MW, Qload_Mvar → Carga conectada na barra

- Slack → Flag para indicar se é barra slack

- De, Para → Linhas (origem e destino)

- R_ohm, X_ohm, B → Parâmetros das linhas

- Comprimento_km → Comprimento da linha

- Imax_kA → Corrente máxima permitida na linha


👉 Transformar esse CSV direto em 3 DataFrames separados para o Pandapower:

df_barras

df_linhas

df_cargas

E depois alimentar o seu código de rede.

In [46]:
import pandapower as pp
import numpy as np
import plotly.graph_objects as go
from IPython.display import display



def show_df_info(df, debug = False):
    print(f"\n🔍 Informações do DataFrame:")
    print(f"Colunas: {list(df.columns)}")
    display(df.head())
    print(f"Dimensões: {df.shape}")

    if debug:
        print(f"Tipos de Dados:\n{df.dtypes}")

def validar_dataframe(df, campos_obrigatorios, nome_df):
    for campo in campos_obrigatorios:
        if campo not in df.columns:
            raise ValueError(f"Mestre Pedro Victor, o campo '{campo}' é obrigatório no DataFrame '{nome_df}' e não foi encontrado.")
    return True


def analyze_contingency_pandapower(net):
    linhas_criticas = []

    # Desativa cada linha e verifica se o sistema ainda opera dentro dos limites
    net.line['in_service'] = True  # Garante que todas as linhas estão inicialmente ativas
    for linha in net.line.index:

        # Desativa a linha
        net.line.loc[linha, 'in_service'] = False
        
        try:
            # Executa o fluxo de Potencia
            pp.runpp(net )

            # Verifica se as tensões estão dentro dos limites e se as linhas estão sobrecarregadas
            if net.res_bus.vm_pu.max() > 1.05 or net.res_bus.vm_pu.min() < 0.95 or net.res_line.loading_percent.max() > 100:
                linhas_criticas.append(linha)
        except:
            print(f"Erro ao calcular o fluxo de carga com a linha {linha} desativada.")
        
        finally:
            # Reativa a linha para o próximo teste
            net.line.loc[linha, 'in_service'] = True
    
    
    return linhas_criticas

def prepara_tabelas_anarede(df):
    # BARRAS
    df_barras = df[["Número", "Nome Barra", "Tensão Base (kV)"]].drop_duplicates().rename(columns={
        "Número": "id",
        "Nome Barra": "nome",
        "Tensão Base (kV)": "tensao"
    }).reset_index(drop=True)

    # CARGAS
    df_cargas = df[["Número", "Carga Ativa (MW)", "Carga Reativa (Mvar)"]].dropna().rename(columns={
        "Número": "bus",
        "Carga Ativa (MW)": "p_mw",
        "Carga Reativa (Mvar)": "q_mvar"
    }).reset_index(drop=True)
    df_cargas["nome"] = "Carga_" + df_cargas["bus"].astype(str)

    # SLACK (maior gerador)
    # Garantir que a coluna está numérica
    df["Geração Ativa (MW)"] = pd.to_numeric(df["Geração Ativa (MW)"], errors='coerce')

    # Agora identifica a barra com maior geração
    df_slack = df.loc[df["Geração Ativa (MW)"] == df["Geração Ativa (MW)"].max(), ["Número"]].rename(columns={"Número": "id"}).reset_index(drop=True)


    # LINHAS — ⚠️ você precisa ter essas colunas no DataFrame!
    if {"De", "Para", "R_ohm", "X_ohm", "Comprimento_km", "Imax_kA"}.issubset(df.columns):
        df_linhas = df[["De", "Para", "R_ohm", "X_ohm", "Comprimento_km", "Imax_kA"]].rename(columns={
            "De": "from_bus",
            "Para": "to_bus",
            "R_ohm": "r_ohm_per_km",
            "X_ohm": "x_ohm_per_km",
            "Comprimento_km": "length_km",
            "Imax_kA": "max_i_ka"
        }).reset_index(drop=True)
        df_linhas["c_nf_per_km"] = 0
    else:
        df_linhas = pd.DataFrame()  # vazio, se não tiver DLIN junto

    return {
        "barras": df_barras,
        "linhas": df_linhas,
        "cargas": df_cargas,
        "slack": df_slack
    }



def criar_rede_parametrizada(df_barras, df_linhas, df_cargas, df_slack):
    net = pp.create_empty_network()

    # Barras
    validar_dataframe(df_barras, ["id", "nome", "tensao"], "Barras")
    for _, barra in df_barras.iterrows():
        pp.create_bus(net, name=barra["nome"], vn_kv=barra["tensao"], index=barra["id"])

    # Slack (primeira barra da lista slack)
    if not df_slack.empty:
        slack_bus = df_slack.iloc[0]["id"]
        slack_nome = df_barras[df_barras["id"] == slack_bus]["nome"].values[0]
        pp.create_ext_grid(net, bus=slack_bus, vm_pu=1.0, name=f"Slack - {slack_nome}")

    # Linhas
    validar_dataframe(df_linhas, ["from_bus", "to_bus", "length_km", "r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka"], "Linhas")
    for _, linha in df_linhas.iterrows():
        pp.create_line_from_parameters(
            net,
            from_bus=linha["from_bus"],
            to_bus=linha["to_bus"],
            length_km=linha["length_km"],
            r_ohm_per_km=linha["r_ohm_per_km"],
            x_ohm_per_km=linha["x_ohm_per_km"],
            c_nf_per_km=linha["c_nf_per_km"],
            max_i_ka=linha["max_i_ka"],
            name=f"Linha {linha['from_bus']}->{linha['to_bus']}"
        )

    # Cargas
    validar_dataframe(df_cargas, ["bus", "p_mw", "q_mvar", "nome"], "Cargas")
    for _, carga in df_cargas.iterrows():
        pp.create_load(net, bus=carga["bus"], p_mw=carga["p_mw"], q_mvar=carga["q_mvar"], name=carga["nome"])

    pp.runpp(net)
    return net


def exibir_resultados(net, titulo):
    print(f"\n🔎 {titulo} - Tensões nas Barras:")
    display(net.res_bus)

    print(f"\n🔎 {titulo} - Correntes nas Linhas:")
    display(net.res_line)

    print(f"\n🔎 {titulo} - Potências nas Cargas:")
    display(net.res_load)






In [52]:
if __name__ == "__main__":
    caminho = r"C:\Users\pedrovictor.veras\Documents\GitHub\Repopulation-With-Elite-Set\src\RedeEletrica\assets\sistema_16_barras.csv"
    df_raw = pd.read_csv(caminho, encoding="latin1", sep=";", on_bad_lines="skip")

    #display(df_raw.head())
    print(df_raw["Grupo Base"].value_counts())

    tabelas = prepara_tabelas_anarede(df_raw)

    df_barras = tabelas["barras"]
    df_linhas = tabelas["linhas"]
    df_cargas = tabelas["cargas"]
    df_slack  = tabelas["slack"]


    # Exibindo informações dos DataFrames
    #show_df_info(df_raw)

    print(df_linhas)



    # Criando minha Rede
    net = criar_rede_parametrizada(df_barras, df_linhas, df_cargas, df_slack)
    exibir_resultados(net, "Caso Base")


Grupo Base
F    14
B    12
C     8
A     8
E     5
D     1
Name: count, dtype: int64
Empty DataFrame
Columns: []
Index: []


ValueError: Mestre Pedro Victor, o campo 'from_bus' é obrigatório no DataFrame 'Linhas' e não foi encontrado.

## Simulação com pandapower

In [ ]:
from rede_eletrica import RedeEletricaPandaPower

import pandas as pd
import pandapower as pp
import pandapower.networks as ppnets
import pandapower.plotting as ppl
from pandapower.plotting.plotly import pf_res_plotly

import matplotlib.pyplot as mplt
import numpy as np

# Supondo que RedeEletricaPandaPower seja uma classe definida
network_name = "14"
rede = RedeEletricaPandaPower(network_name, debug=True)
net = rede.net
exibir_tabelas = False

# Aumentar o estresse na rede significa que mais linhas se tornam críticas,
# o que é bom para fins de demonstração da análise de contingência.

# Desativar a subestação externa (external grid)
net.ext_grid['in_service'] = False

# Aumentar a carga padrão para estressar ainda mais a rede:
net.load.scaling = 1.5

# Ajustar as tensões terminais dos geradores para que as tensões dos barramentos e estejam dentro de uma faixa mais prática, de 0.95 pu a 1.05 pu.
net.gen['vm_pu'] = 1.045

# Realizar um despacho simples de geradores maximizando os três primeiros geradores e definindo o quarto como slack.
net.gen.loc[0, 'p_mw'] = 120
net.gen.loc[1, 'p_mw'] = 100
net.gen.loc[2, 'p_mw'] = 100
net.gen.loc[3, 'slack'] = True

# Executar o Fluxo de Potência na condição base
pp.runpp(net, numba=False)
print("Fluxo de Potencia executado!")

# Imprimir a Geração Total e a Carga como uma Verificação Rápida
gen_mw_total = net.res_gen['p_mw'].sum()
imports_mw_total = net.res_ext_grid['p_mw'].sum()

print('Geração total em MW:', gen_mw_total + imports_mw_total)
print('Geração total importada em MW:', imports_mw_total)
print('Geração total local em MW:', gen_mw_total)
print('Carga total em MW:', net.res_load['p_mw'].sum())


# Exibir tabelas se solicitado
if exibir_tabelas:
                print("\n--- Tabelas Detalhadas da Rede ---")
                print(f"Nome da Rede: IEEE {network_name}")
                print("\nEstrutura Completa da Rede (Objeto 'net'):")
                print(net)

                if 'gen' in net:
                    print("\nTabela de Geradores ('net.gen'):")
                    display(net.gen)


                if 'load' in net:
                    print("\nTabela de Cargas ('net.load'):")
                    display(net.load)


                if 'res_gen' in net:
                     print("\nTabela de Resultados dos Geradores ('net.res_gen'):")
                     display(net.res_gen)


                if 'res_bus' in net:
                    print("\nTabela de Resultados dos Barramentos ('net.res_bus'):")
                    display(net.res_bus)


                # Adicionar outras tabelas comuns se existirem na sua rede
                if 'line' in net:
                     print("\nTabela de Linhas ('net.line'):")
                     display(net.line)


                if 'res_line' in net:
                     print("\nTabela de Resultados das Linhas ('net.res_line'):")
                     display(net.res_line)


                if 'trafo' in net:
                     print("\nTabela de Transformadores ('net.trafo'):")
                     display(net.trafo)

                if 'res_trafo' in net:
                     print("\nTabela de Resultados dos Transformadores ('net.res_trafo'):")
                     display(net.res_trafo)


                print("\n--- Fim das Tabelas ---")


In [ ]:

# --- Análise de Contingência para encontrar linhas críticas ---

def realizar_analise_contingencia(rede, vmax=1.05, vmin=0.95, line_loading_max=100):
    """
    Realiza a análise de contingência para cada linha na rede
    e retorna os índices das linhas críticas.
    """
    linhas = rede.line.index
    indices_linhas_criticas = []

    print("\nRealizando análise de contingência para as linhas...")
    for l in linhas:

        # Temporariamente desativar a linha (simulando a contingência)
        rede.line.loc[l, 'in_service'] = False
        try:
            # Executar o fluxo de potência com a contingência
            pp.runpp(rede, numba=False)

            # Verificar violações (limites de tensão e carregamento de linha)
            if rede.res_bus.vm_pu.max() > vmax or rede.res_bus.vm_pu.min() < vmin or rede.res_line.loading_percent.max() > line_loading_max:
                indices_linhas_criticas.append(l)

        except pp.LoadflowNotConverged:
            print(f"Fluxo de potência não convergiu para a contingência da linha {l}. Considerada crítica.")
            indices_linhas_criticas.append(l)
        except Exception as e:
            print(f"Ocorreu um erro durante o fluxo de potência para a contingência da linha {l}: {e}")
            # Dependendo dos requisitos da sua análise, você pode querer tratar outros erros como críticos
            # indices_linhas_criticas.append(l)
        finally:
            # Sempre retornar a linha ao serviço
            rede.line.loc[l, 'in_service'] = True

            # Executar o fluxo de potência na condição base novamente se o fluxo de potência falhou durante a contingência
            pp.runpp(rede, numba=False)

    return list(set(indices_linhas_criticas)) # Remover duplicatas



# --- Plotagem com cores de status ---
def plotar_rede_com_status(rede, indices_linhas_criticas):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    bus_color = ['green' if status else 'gray' for status in rede.bus.in_service]

    line_color = []

    for i in rede.line.index:
        if not rede.line.at[i, 'in_service'] or i in indices_linhas_criticas:
            line_color.append('orange')
        else:
            line_color.append('blue')

    trafo_color = ['green' if status else 'gray' for status in rede.trafo.in_service] if not rede.trafo.empty else None



    ax = ppl.simple_plot(
        rede,

        bus_color=bus_color,
        line_color=line_color,
        trafo_color=trafo_color,
        ext_grid_color='black',


        ext_grid_size=3.0,
        line_width=2.0,
        bus_size=3,
        trafo_size=4,
        switch_size = 4,
        bus_dc_size = 4,

        plot_sgens = True,
        plot_line_switches=True,
        plot_loads = True,
        respect_switches = True,


    )

    legenda = [
        Line2D([0], [0], marker='o', color='w', label='Barramento Ativo',
               markerfacecolor='green', markersize=10),
        Line2D([0], [0], color='blue', lw=2, label='Linha Normal'),
        Line2D([0], [0], color='orange', lw=2, linestyle='--', label='Linha Crítica/Inativa'),
        Line2D([0], [0], color='green', lw=2, label='Trafo Ativo'),
        Line2D([0], [0], color='gray', lw=2, linestyle='--', label='Trafo Inativo'),
    ]

    ax.legend(handles=legenda, loc='best')
    #ax.set_title("Status da Rede Elétrica IEEE 14 - Inclusiva")
    ax.set_xlabel("Eixo X")
    ax.set_ylabel("Eixo Y")
    ax.grid(True)

    plt.show()



# Executar a análise de contingência
print(net)
critical_lines_indx = realizar_analise_contingencia(net)
print(f"Índices das linhas críticas = {critical_lines_indx} ")

# Executar a plotagem
plotar_rede_com_status(net, critical_lines_indx)

print("\n\nPlot da Rede com cores de status pela tensão nos barramentos")
print("Laranja - Trafos")
print("Amarelo - Barramentos (Bus Voltage in pu)")
print("Azul - Linhas")
mapas = ["streets", "light", "dark", "satellite"]
ppl.pf_res_plotly(net,  line_width= 3, bus_size= 50, map_style= mapas[2], figsize= 1, filename= "network.html", on_map=False, projection='epsg:8859')
display(net.res_bus)

## Simulação com dados de TXT em Excel

In [ ]:
import pandas as pd
import pandapower as pp
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


def carregar_dados_excel(caminho_arquivo):
    print(f"📥 Lendo dados do Excel: {caminho_arquivo}")
    sheets = pd.ExcelFile(caminho_arquivo).sheet_names
    dados = {}
    for sheet in sheets:
        dados[sheet] = pd.read_excel(caminho_arquivo, sheet_name=sheet, index_col=0)
    return dados


def criar_rede_com_dados(dados):
    net = pp.create_empty_network()

    # Barras
    if "bus" not in dados:
        raise ValueError("Mestre Pedro Victor, sheet 'bus' é obrigatória!")
    for idx in dados["bus"].index:
        row = dados["bus"].loc[idx]
        pp.create_bus(net, vn_kv=row["vn_kv"], name=row.get("name", f"Bus {idx}"))

    # Slack - Barra de Referência
    if "slack" in dados:
        for idx in dados["slack"].index:
            row = dados["slack"].loc[idx]
            pp.create_ext_grid(net, bus=row["bus"], vm_pu=row["vm_pu"], va_degree=row["va_degree"])

    # Cargas - Geradores
    if "load" not in dados:
        raise ValueError("Mestre, sheet 'load' também é obrigatória!")
    for idx in dados["load"].index:
        row = dados["load"].loc[idx]
        pp.create_load(net, bus=row["bus"], p_mw=row["p"], q_mvar=row.get("q", 0), name=f"Load {idx}")

    # Linhas
    if "line" not in dados:
        raise ValueError("Mestre, sheet 'line' também é obrigatória!")
    for idx in dados["line"].index:
        row = dados["line"].loc[idx]
        pp.create_line_from_parameters(
            net,
            from_bus=row["from_bus"],
            to_bus=row["to_bus"],
            length_km=row["length_km"],
            r_ohm_per_km=row["r_ohm_per_km"],
            x_ohm_per_km=row["x_ohm_per_km"],
            c_nf_per_km=row["c_nf_per_km"],
            max_i_ka=row["max_i_ka"],
            name=f"Line {idx}"
        )

    # Transformadores (se houver)
    if "trafo" in dados:
        for idx in dados["trafo"].index:
            row = dados["trafo"].loc[idx]
            pp.create_transformer(net, hv_bus=row["hv_bus"], lv_bus=row["lv_bus"], std_type=row["std_type"])

    return net


def exibir_resultados(net, titulo):
    print(f"\n🔎 {titulo} - Tensões nas Barras:")
    display(net.res_bus)

    print(f"\n🔎 {titulo} - Correntes nas Linhas:")
    display(net.res_line)

    print(f"\n🔎 {titulo} - Potências nas Cargas:")
    display(net.res_load)


def simular_contingencia(net, linha_out):
    net_contingencia = net.deepcopy()
    pp.create_switch(net_contingencia, bus=net_contingencia.line.from_bus.at[linha_out],
                     element=linha_out, et='l', closed=False, name=f"Contingência Linha {linha_out}")
    pp.runpp(net_contingencia)

    sobrecarga = net_contingencia.res_line.i_ka > net_contingencia.line.max_i_ka
    print(f"\n⚡ Contingência: Linha {linha_out} desligada")
    print("\nLinhas com sobrecarga:")
    display(net_contingencia.res_line[sobrecarga])

    return net_contingencia, sobrecarga


def definir_cor_tensao(vn_kv):
    if vn_kv < 69:
        return 'black'
    elif vn_kv <= 69:
        return 'yellow'
    elif vn_kv <= 88:
        return 'orange'
    elif vn_kv <= 139:
        return 'blue'
    elif vn_kv <= 230:
        return 'green'
    elif vn_kv <= 440:
        return 'purple'
    else:
        return 'gray'


def plotar_linhas_por_tensao(net, titulo, sobrecarga=None):
    cores = []
    for idx in net.line.index:
        vn_kv = max(net.bus.vn_kv[net.line.from_bus[idx]], net.bus.vn_kv[net.line.to_bus[idx]])
        cor = definir_cor_tensao(vn_kv)

        # Se for sobrecarga, sobrepõe a cor para vermelho
        if sobrecarga is not None and sobrecarga.at[idx]:
            cor = 'red'
        cores.append(cor)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=net.res_line.index,
        y=net.res_line.i_ka,
        marker_color=cores,
        name="Corrente nas Linhas"
    ))

    fig.update_layout(
        title=titulo,
        xaxis_title="Linhas",
        yaxis_title="Corrente (kA)"
    )

    fig.show()


def main():
    # Caminho para seu Excel
    caminho_excel = "dados_rede_passarinhos.xlsx"

    # Carregar dados
    dados = carregar_dados_excel(caminho_excel)

    # Criar a rede
    net = criar_rede_com_dados(dados)

    # Fluxo de potência base
    pp.runpp(net)
    exibir_resultados(net, titulo="Caso Base")

    # Plot base
    plotar_linhas_por_tensao(net, titulo="Correntes nas Linhas - Caso Base")

    # Contingência: exemplo, desligando linha 1
    net_contingencia, sobrecarga = simular_contingencia(net, linha_out=1)
    exibir_resultados(net_contingencia, titulo="Pós Contingência")

    # Plot com sobrecarga destacada
    plotar_linhas_por_tensao(net_contingencia, titulo="Correntes nas Linhas - Pós Contingência (Sobrecarga em Vermelho)", sobrecarga=sobrecarga)


if __name__ == "__main__":
    main()
